# 01 — Data Loading

Download NHANES 2015–2016 and 2017–2020 (pre-pandemic) cycles, merge, clean, and save analysis-ready parquet files.

In [ ]:
from pathlib import Path
from functools import reduce
import urllib.request
import numpy as np
import pandas as pd
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print('Imports OK')

## 1. Configuration

In [ ]:
BASE_URL  = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public'
RAW_DIR_I = Path('../data/raw/cycle_I')
RAW_DIR_J = Path('../data/raw/cycle_J')
PROCESSED = Path('../data/processed')

for d in [RAW_DIR_I, RAW_DIR_J, PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)

PHQ_ITEMS   = [f'DPQ0{i}0' for i in range(1, 10)]
CONFOUNDERS = ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'SMQ040_bin', 'ALQ130', 'RIDRETH3']
MODULES = {
    'DEMO':   ('DEMO_I.XPT',   'DEMO_J.XPT'),
    'BMX':    ('BMX_I.XPT',    'BMX_J.XPT'),
    'HSCRP':  ('HSCRP_I.XPT',  'HSCRP_J.XPT'),
    'CBC':    ('CBC_I.XPT',    'CBC_J.XPT'),
    'BIOPRO': ('BIOPRO_I.XPT', 'BIOPRO_J.XPT'),
    'FERTIN': ('FERTIN_I.XPT', 'FERTIN_J.XPT'),
    'DPQ':    ('DPQ_I.XPT',    'DPQ_J.XPT'),
    'SMQ':    ('SMQ_I.XPT',    'SMQ_J.XPT'),
    'ALQ':    ('ALQ_I.XPT',    'ALQ_J.XPT'),
}
print('Config done.')

## 2. Download

In [ ]:
def download_xpt(dest, *urls):
    if dest.exists():
        print(f'  [cached] {dest.name}')
        return
    for url in urls:
        try:
            urllib.request.urlretrieve(url, dest)
            print(f'  [OK]     {dest.name}  ({dest.stat().st_size//1024} KB)')
            return
        except Exception:
            continue
    print(f'  [WARN]   {dest.name} not found')

print('Cycle I (2015-2016):')
for mod, (fi, _) in MODULES.items():
    download_xpt(RAW_DIR_I / fi, f'{BASE_URL}/2015/DataFiles/{fi}')

print()
print('Cycle J (2017-2020):')
for mod, (_, fj) in MODULES.items():
    download_xpt(RAW_DIR_J / fj, f'{BASE_URL}/2017/DataFiles/{fj}')

## 3. Load and Merge

In [ ]:
def load_cycle(raw_dir, suffix):
    def read(filename, usecols):
        path = raw_dir / filename
        if not path.exists():
            print(f'  [WARN] {filename} missing')
            return pd.DataFrame(columns=usecols)
        df, _ = pyreadstat.read_xport(str(path), usecols=usecols)
        return df

    frames = [
        read(f'DEMO_{suffix}.XPT',  ['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVPSU','SDMVSTRA']),
        read(f'BMX_{suffix}.XPT',   ['SEQN','BMXBMI']),
        read(f'HSCRP_{suffix}.XPT', ['SEQN','LBXHSCRP']),
        read(f'CBC_{suffix}.XPT',   ['SEQN','LBXWBCSI','LBXNEPCT','LBXLYPCT']),
        read(f'BIOPRO_{suffix}.XPT',['SEQN','LBXSAL']),
        read(f'FERTIN_{suffix}.XPT',['SEQN','LBXFER']),
        read(f'DPQ_{suffix}.XPT',   ['SEQN'] + PHQ_ITEMS),
        read(f'SMQ_{suffix}.XPT',   ['SEQN','SMQ040']),
        read(f'ALQ_{suffix}.XPT',   ['SEQN','ALQ130']),
    ]
    merged = reduce(lambda l, r: l.merge(r, on='SEQN', how='left'), frames)
    merged['cycle'] = suffix
    print(f'  Cycle {suffix}: N = {len(merged):,}')
    return merged

df_I = load_cycle(RAW_DIR_I, 'I')
df_J = load_cycle(RAW_DIR_J, 'J')

## 4. Pool Cycles

In [ ]:
df_all = pd.concat([df_I, df_J], ignore_index=True)

# Cast Arrow/StringDtype columns → plain object (fastparquet compatibility)
for col in df_all.columns:
    if isinstance(df_all[col].dtype, pd.StringDtype) or str(df_all[col].dtype) == 'str':
        df_all[col] = df_all[col].astype(object)

print(f'Pooled N = {len(df_all):,}')

## 5. Survey Weights

`WTPOOL = WTMEC2YR / 2` — standard CDC method for pooling two 2-year cycles.

In [ ]:
df_all['WTPOOL'] = df_all['WTMEC2YR'] / 2
df_all.loc[df_all['WTMEC2YR'].isna() | (df_all['WTMEC2YR'] == 0), 'WTPOOL'] = 0
print(f'MEC-examined (weight > 0): {(df_all["WTPOOL"] > 0).sum():,}')

## 6. Variable Engineering

### 6a. PHQ-9

In [ ]:
for col in PHQ_ITEMS:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')
    df_all.loc[df_all[col].isin([7, 9]), col] = np.nan

df_all['PHQ9_total'] = df_all[PHQ_ITEMS].sum(axis=1, skipna=False)
df_all['depression'] = np.where(df_all['PHQ9_total'].isna(), np.nan,
                                (df_all['PHQ9_total'] >= 10).astype(float))
print(f'PHQ-9 complete: {df_all["PHQ9_total"].notna().sum():,}')
print(f'Depressed:      {(df_all["depression"]==1).sum():,}')

### 6b. Biomarkers

In [ ]:
df_all['NLR']          = df_all['LBXNEPCT'] / df_all['LBXLYPCT']
df_all.loc[df_all['LBXLYPCT'] == 0, 'NLR'] = np.nan
df_all['log_CRP']      = np.log(df_all['LBXHSCRP'] + 1)
df_all['log_ferritin'] = np.log(df_all['LBXFER'])
df_all['log_NLR']      = np.log(df_all['NLR'])
df_all['log_WBC']      = np.log(df_all['LBXWBCSI'])
df_all[['log_CRP','log_WBC','log_NLR','log_ferritin']].describe().round(3)

### 6c. Confounders

In [ ]:
df_all['SMQ040'] = pd.to_numeric(df_all['SMQ040'], errors='coerce')
df_all.loc[df_all['SMQ040'].isin([7, 9]), 'SMQ040'] = np.nan
df_all['SMQ040_bin'] = np.where(df_all['SMQ040'].isna(), np.nan,
                                df_all['SMQ040'].isin([1, 2]).astype(float))

df_all['ALQ130'] = pd.to_numeric(df_all['ALQ130'], errors='coerce')
df_all.loc[df_all['ALQ130'].isin([777, 999]), 'ALQ130'] = np.nan

for col in ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'RIDRETH3']:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

print('Confounders cleaned.')

## 7. Missing Data Audit

In [ ]:
key_vars = (['LBXHSCRP','LBXWBCSI','LBXNEPCT','LBXLYPCT','LBXSAL','LBXFER'] +
            ['NLR','log_CRP','log_ferritin','log_NLR','log_WBC'] +
            PHQ_ITEMS + ['PHQ9_total','depression'] +
            CONFOUNDERS + ['WTPOOL'])

missing = pd.DataFrame({
    'n_present':   [df_all[v].notna().sum() for v in key_vars],
    'pct_missing': [df_all[v].isna().mean() * 100 for v in key_vars],
}, index=key_vars).sort_values('pct_missing', ascending=False)
missing['flag'] = missing['pct_missing'] > 20

missing.to_csv(PROCESSED / 'missing_data_report.csv')
print(missing.to_string())

## 8. Complete-Case Split and Save

In [ ]:
PRIMARY_COLS = (
    ['SEQN','WTPOOL','SDMVPSU','SDMVSTRA','cycle','PHQ9_total','depression'] +
    PHQ_ITEMS +
    ['LBXHSCRP','log_CRP','LBXWBCSI','log_WBC','LBXNEPCT','LBXLYPCT',
     'NLR','log_NLR','LBXSAL','LBXFER','log_ferritin'] +
    CONFOUNDERS
)

df_complete = df_all[PRIMARY_COLS].dropna()
print(f'Pooled N:        {len(df_all):,}')
print(f'Complete-case N: {len(df_complete):,}')
print(f'Depression:      {df_complete["depression"].mean()*100:.1f}%')

df_all.to_parquet(PROCESSED / 'nhanes_merged_raw.parquet', engine='fastparquet', index=False)
df_complete.to_parquet(PROCESSED / 'nhanes_analysis_ready.parquet', engine='fastparquet', index=False)
print('Saved.')